# Outlier detectors revision: 5 seeds

In [1]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt
from abc import abstractmethod
from typing import List, Callable, Union, Any, TypeVar, Tuple
from itertools import cycle
Tensor = TypeVar('torch.tensor')
import scipy.io as sio
import os
import math

import numpy as np
import scipy.io as scio
from torch import optim, nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm import tqdm

from torchvision import transforms
import torchvision.utils as vutils

import matplotlib
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, Normalizer
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor, NearestNeighbors
from sklearn.metrics import accuracy_score, adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score, v_measure_score
from scipy.optimize import linear_sum_assignment
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from copy import deepcopy
import time

experiment_seeds = [42, 1, 2, 3, 6]

# Dataset

In [ ]:
'''use a numerical dataset to test the model
label  damaged_floor     damage_extent         number_of_samples
0           0            0%                     1500
1           1            3%                     500
2           1            6%                     500
3           1            10%                    500
4           1,3          5%,10%                 500
5           1,3,5        5%,10%,15%             500
6           2,4,6        10%,15%,20%            500
7           1,3,5,7      10%,15%,20%,25%        500


'''
# Read data from CSV
features = pd.read_csv("TF_mag_numerical_8class_1000features_20dB.csv")
features = features.astype("float32")
# # # Convert DataFrame to PyTorch tensors
X = torch.tensor(features.values[:,:])
X = X.t()

input_dim = X.shape[1]
print(X.shape)


a = []
for i in range(1500):
    a.append(0)
for i in np.arange(1500,2000):
    a.append(1)
for i in np.arange(2000,2500):
    a.append(2)
for i in np.arange(2500,3000):
    a.append(3)
for i in np.arange(3000,3500):
    a.append(4)
for i in np.arange(3500,4000):
    a.append(5)
for i in np.arange(4000,4500):
    a.append(6)
for i in np.arange(4500,5000):
    a.append(7)
print(len(a))

y0 = torch.tensor(a)
y0 = y0.unsqueeze(1)
y = [int(_) for _ in y0]
y = torch.tensor(y)
num_classes = y.max().item() + 1
print(f"number of total classes: {num_classes}")

plt.plot(y)
plt.show()

# Base PCA

In [3]:
pca_components = 10
pca = PCA(n_components=pca_components)
data_pca_base = pca.fit_transform(X).astype("float32")
print('PCA data shape:', data_pca_base.shape)

PCA data shape: (5000, 10)


# VAE model and helpers

In [4]:
class VAE(nn.Module):

    def __init__(self, input_dim: int, hidden_layers: List[int], latent_dim: int, output_type: str = 'linear'):
        
        super(VAE, self).__init__()
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.latent_dim = latent_dim
        self.output_type = output_type
    
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, hidden_layers[0]),
            nn.ReLU(),
            nn.Linear(hidden_layers[0], hidden_layers[1]),
            nn.ReLU(),
            nn.Linear(hidden_layers[1], hidden_layers[2]),
            nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(hidden_layers[2], latent_dim)
        self.fc_log_var = nn.Linear(hidden_layers[2], latent_dim)

        # Build Decoder

        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, hidden_layers[2]),
            nn.ReLU(),
            nn.Linear(hidden_layers[2], hidden_layers[1]),
            nn.ReLU(),
            nn.Linear(hidden_layers[1], hidden_layers[0]),
            nn.ReLU(),
            nn.Linear(hidden_layers[0], self.input_dim)
        )

    def encode(self, input: Tensor) -> List[Tensor]:
        """
        Encodes the input by passing through the encoder network and returns the latent codes.
        param input: (Tensor) Input tensor to encoder [N x C x H x W]
        return: (Tensor) List of latent codes
        """
        result = self.encoder(input)
        mu = self.fc_mu(result)
        log_var = self.fc_log_var(result)
        return [mu, log_var]

    def decode(self, z: Tensor) -> Tensor:
        """
        Maps the given latent codes onto the image space.
        param z: (Tensor) [B x D]
        return: (Tensor) [B x C x H x W]
        """

        result = self.decoder(z)

        if self.output_type == 'linear':
            pass
        elif self.output_type == 'sigmoid':
            result = torch.sigmoid(result)
        else: # tahn
            result = torch.tanh(result)

        return result
    
    def reparameterize(self, mu: Tensor, log_var: Tensor) -> Tensor:
        """
        Reparameterization trick to sample from N(mu, var) from
        N(0,1).
        :param mu: (Tensor) Mean of the latent Gaussian [B x D]
        :param logvar: (Tensor) Standard deviation of the latent Gaussian [B x D]
        :return: (Tensor) [B x D]
        """
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return eps * std + mu  
      
    # Forward propagation: inference network outputs mu(x) and log_var(x) given an input x, then use repameterization trick to sample a batch of z, which are feed to DPGMM model
    def forward(self, input: Tensor, **kwargs) -> List[Tensor]:
        mu, log_var = self.encode(input)
        z = self.reparameterize(mu, log_var)
        return  [self.decode(z), input, mu, log_var, z] # [recon, input, mu, log_var, z]

    def reconstruction(self, x: Tensor, **kwargs) -> Tensor:
        """
        Given an input sample x, returns the reconstructed sample
        :param x: (Tensor) [B x C x H x W]
        :return: (Tensor) [B x C x H x W]
        """

        return self.forward(x)[0]

    def loss_function(self, *args, **kwargs) -> dict:
        recons = args[0]
        input = args[1]
        mu = args[2] # u(z|x)
        log_var = args[3] # sigma(z|x)
        z = args[4]  # batch_size * latent_dim

        # reconstruction loss
        recons_loss = F.mse_loss(recons, input, reduction='sum')
        kld_loss = torch.mean(-0.5 * torch.sum(1 + log_var - mu ** 2 - log_var.exp(), dim = 1), dim = 0)
        loss = recons_loss + kld_loss

        return {'loss': loss, 'reconstruction_loss':recons_loss, 'kld_loss': kld_loss, 'z': z}

In [5]:
def move_dict_to_device(state_dict, device):
    '''move all tensors in state_dict to the device'''
    new_state_dict = {}
    for k, v in state_dict.items():
        if isinstance(v, torch.Tensor):
            new_state_dict[k] = v.to(device)
        else:
            new_state_dict[k] = v
    return new_state_dict

def forward(model, input: Tensor, **kwargs) -> Tensor:
    return model(input, **kwargs)
    
    
def training_step(model, batch_samples, batch_labels):
    curr_device = batch_samples.device

    batch_size = batch_samples.size(0)
    results = forward(model, batch_samples, labels = batch_labels)
    train_loss = model.loss_function(*results, batch_size = batch_size)
    train_loss.update({'labels': batch_labels})
    return train_loss    # latent encoding

def validation_step(model, batch_samples, batch_labels):
    batch_size = batch_samples.size(0)
    results = forward(model, batch_samples, labels = batch_labels)
    val_loss = model.loss_function(*results, batch_size = batch_size)
                                        
    val_loss.update({'labels': batch_labels})
    return val_loss


# Detector functions

In [6]:
def compute_detection_metrics(y_pred, healthy_count=1500, normal_value=0, anomaly_value=1):
    fn, fp = 0, 0
    for i in range(len(y_pred)):
        if i <= healthy_count and y_pred[i] == anomaly_value:
            fp += 1
        elif i > healthy_count and y_pred[i] == normal_value:
            fn += 1
    dda = 1 - (fn + fp) / len(y_pred)
    return dda, fp, fn


def run_vae_msd_detector(seed, data_pca_base, y, total_epochs=300, batch_size=32):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_model = torch.tensor(data_pca_base)
    input_dim = X_model.shape[1]
    print('input_dim:', input_dim)

    dataset = TensorDataset(X_model, y)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = VAE(input_dim=input_dim, hidden_layers=[500, 500, 2000], latent_dim=10, output_type='linear').to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-3)

    training_loss_epoch = []
    recon_loss_epoch = []
    kl_loss_epoch = []
    val_loss_epoch = []

    train_start_time = time.time()
    for epoch in range(total_epochs):
        pbar = tqdm(train_loader, desc=f"current_epoch {epoch + 1}/{total_epochs}")
        training_loss_batch = []
        recon_loss_batch = []
        kl_loss_batch = []

        model.train()
        for batch_samples, batch_labels in pbar:
            batch_samples = batch_samples.to(device)
            train_loss = training_step(model, batch_samples, batch_labels)

            loss_tensor = train_loss['loss']
            training_loss_batch.append(train_loss['loss'].item())
            recon_loss_batch.append(train_loss['reconstruction_loss'].item())
            kl_loss_batch.append(train_loss['kld_loss'].item())

            optimizer.zero_grad()
            loss_tensor.backward()
            optimizer.step()

        training_loss_epoch.append(np.mean(training_loss_batch))
        recon_loss_epoch.append(np.mean(recon_loss_batch))
        kl_loss_epoch.append(np.mean(kl_loss_batch))

        print(
            f"training_loss:{np.mean(training_loss_batch):.4f}",
            f"reconstruction_loss:{np.mean(recon_loss_batch):.4f}",
            f"kl_loss:{np.mean(kl_loss_batch):.4f}",
        )

        with torch.no_grad():
            model.eval()
            valid_loss_batch = []
            val_recon_loss_batch = []
            val_kl_loss_batch = []

            for batch_samples, batch_labels in test_loader:
                batch_samples = batch_samples.to(device)
                val_loss = validation_step(model, batch_samples, batch_labels)
                valid_loss_batch.append(val_loss['loss'].item())
                val_recon_loss_batch.append(val_loss['reconstruction_loss'].item())
                val_kl_loss_batch.append(val_loss['kld_loss'].item())

            val_loss_epoch.append(np.mean(valid_loss_batch))
            print(
                f"valid_loss:{np.mean(valid_loss_batch):.4f}",
                f"val_reconstruction_loss:{np.mean(val_recon_loss_batch):.4f}",
                f"val_kl_loss:{np.mean(val_kl_loss_batch):.4f}",
            )

    model.eval()
    latent_representations = model.encode(X_model.to(device))[0].detach().cpu().numpy()
    training_data = latent_representations[:int(1500 * 0.8), :]
    mu = training_data.mean(axis=0)
    cov = np.cov(training_data, rowvar=False)
    cov_inv = np.linalg.inv(cov)

    MSD = []
    for i in range(latent_representations.shape[0]):
        MSD.append((latent_representations[i] - mu) @ cov_inv @ (latent_representations[i] - mu).T)
    MSD = np.asarray(MSD)

    sorted_MSD = np.sort(MSD[:int(1500 * 0.8)])
    threshold = sorted_MSD[int(1500 * 0.8 * 0.95)]
    y_pred = (MSD > threshold).astype(int)  # 1=anomaly, 0=normal

    dda, fp, fn = compute_detection_metrics(y_pred, normal_value=0, anomaly_value=1)
    runtime = time.time() - train_start_time

    return {
        'detector': 'VAE_MSD',
        'seed': seed,
        'damage_detection_accuracy': dda,
        'false_positive': fp,
        'false_negative': fn,
        'runtime_seconds': runtime,
        'threshold': threshold,
        'scores': MSD,
        'predicted_outlier': y_pred,
        'training_loss_epoch': training_loss_epoch,
        'val_loss_epoch': val_loss_epoch,
    }


def run_ocsvm_detector(seed, data_pca_base):
    torch.manual_seed(seed)
    np.random.seed(seed)

    start_time = time.time()
    X_model = torch.tensor(data_pca_base)
    pca_components = 10
    pca = PCA(n_components=pca_components)
    data_pca = pca.fit_transform(X_model).astype('float32')
    print('PCA data shape:', data_pca.shape)

    ocsvm = OneClassSVM(kernel='rbf')
    ocsvm.fit(data_pca[:int(1500 * 0.8), :])

    y_raw = ocsvm.predict(data_pca)  # +1=inlier, -1=outlier
    y_pred = np.where(y_raw == -1, 1, 0)  # 1=anomaly, 0=normal

    dda, fp, fn = compute_detection_metrics(y_pred, normal_value=0, anomaly_value=1)
    runtime = time.time() - start_time

    return {
        'detector': 'OCSVM',
        'seed': seed,
        'damage_detection_accuracy': dda,
        'false_positive': fp,
        'false_negative': fn,
        'runtime_seconds': runtime,
        'threshold': np.nan,
        'scores': y_raw,
        'predicted_outlier': y_pred,
    }


def run_knn_detector(seed, data_pca_base):
    torch.manual_seed(seed)
    np.random.seed(seed)

    start_time = time.time()
    X_model = torch.tensor(data_pca_base)
    pca_components = 10
    pca = PCA(n_components=pca_components)
    data_pca = pca.fit_transform(X_model).astype('float32')

    X_train = data_pca[:int(1500 * 0.8), :]
    k = 1
    nn_model = NearestNeighbors(n_neighbors=k, radius=1).fit(X_train)
    d_train, _ = nn_model.kneighbors(data_pca, n_neighbors=k, return_distance=True)

    score_train = d_train[:, -1]
    tau = np.quantile(score_train[:int(1500 * 0.8)], 0.95)
    y_pred = (score_train > tau).astype(int)  # 1=anomaly, 0=normal

    dda, fp, fn = compute_detection_metrics(y_pred, normal_value=0, anomaly_value=1)
    runtime = time.time() - start_time

    return {
        'detector': 'KNN',
        'seed': seed,
        'damage_detection_accuracy': dda,
        'false_positive': fp,
        'false_negative': fn,
        'runtime_seconds': runtime,
        'threshold': tau,
        'scores': score_train,
        'predicted_outlier': y_pred,
    }

# Run five seeds

In [ ]:
seed_results = []

for run_id, seed in enumerate(experiment_seeds, start=1):
    print(f"\n===== Seed {seed} ({run_id}/{len(experiment_seeds)}) =====")

    vae_result = run_vae_msd_detector(seed=seed, data_pca_base=data_pca_base, y=y, total_epochs=300, batch_size=32)
    seed_results.append(vae_result)
    print(
        f"VAE_MSD seed {seed}: DDA={vae_result['damage_detection_accuracy']:.6f}, "
        f"FP={vae_result['false_positive']}, FN={vae_result['false_negative']}, "
        f"runtime={vae_result['runtime_seconds']:.2f}s"
    )

    ocsvm_result = run_ocsvm_detector(seed=seed, data_pca_base=data_pca_base)
    seed_results.append(ocsvm_result)
    print(
        f"OCSVM seed {seed}: DDA={ocsvm_result['damage_detection_accuracy']:.6f}, "
        f"FP={ocsvm_result['false_positive']}, FN={ocsvm_result['false_negative']}, "
        f"runtime={ocsvm_result['runtime_seconds']:.2f}s"
    )

    knn_result = run_knn_detector(seed=seed, data_pca_base=data_pca_base)
    seed_results.append(knn_result)
    print(
        f"KNN seed {seed}: DDA={knn_result['damage_detection_accuracy']:.6f}, "
        f"FP={knn_result['false_positive']}, FN={knn_result['false_negative']}, "
        f"runtime={knn_result['runtime_seconds']:.2f}s"
    )

results_df = pd.DataFrame(seed_results)

metric_columns = [
    'damage_detection_accuracy',
    'false_positive',
    'false_negative',
    'runtime_seconds',
]

summary_df = results_df.groupby('detector')[metric_columns].agg(['mean', 'std'])

In [ ]:
print('\nPer-seed detector results')
display(results_df[[
    'detector',
    'seed',
    'damage_detection_accuracy',
    'runtime_seconds',
]])

print('\nMean and sample standard deviation over seeds')
display(summary_df)